# RQ4: Feature Importance for High Default-Risk Prediction

This Kaggle notebook trains a Random Forest classifier and extracts the most important predictors.

**Outputs saved to `/kaggle/working/rq4_feature_importance/`:**
- `RQ4_feature_importance_table.csv`
- `RQ4_feature_importance_figure.pdf`

**Research question:** Which dataset features contribute most to machine-learning predictions of high default risk?

## Run this cell to generate the actual answer, CSV table, and PDF figure

The notebook automatically searches for the dataset file inside `/kaggle/input`. If it cannot find it, set `DATASET_PATH` manually in the code cell.

In [ ]:

# ============================================================
# Common setup: imports, paths, loading, cleaning, preprocessing
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

RANDOM_STATE = 42

# Change this manually only if Kaggle cannot auto-detect your file.
# Example:
# DATASET_PATH = "/kaggle/input/ev-charging-behavior-analysis-and-financial-risk/your_file.csv"
DATASET_PATH = None

def file_has_expected_columns(path):
    """
    Checks whether a candidate file looks like the raw EV dataset rather than
    an output table generated by one of the notebooks.
    """
    try:
        if path.lower().endswith(".csv"):
            cols = pd.read_csv(path, nrows=0).columns.astype(str).str.strip().tolist()
        elif path.lower().endswith((".xlsx", ".xls")):
            cols = pd.read_excel(path, nrows=0).columns.astype(str).str.strip().tolist()
        else:
            return False
    except Exception:
        return False

    required = {"High_Default_Risk", "Charging_Efficiency_Index"}
    return required.issubset(set(cols)) and len(cols) >= 10

def find_dataset_file():
    """
    Finds the raw CSV or Excel dataset file automatically.
    Works in Kaggle by searching /kaggle/input first.
    Also avoids accidentally selecting CSV output tables generated by the notebooks.
    """
    if DATASET_PATH is not None and os.path.exists(DATASET_PATH):
        return DATASET_PATH

    roots = ["/kaggle/input", ".", "/mnt/data"]
    extensions = (".csv", ".xlsx", ".xls")
    candidates = []

    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            # Skip common output folders when running locally
            if any(part.startswith("rq") and "output" in part for part in dirpath.lower().split(os.sep)):
                continue
            for filename in filenames:
                lower = filename.lower()
                if lower.endswith(extensions):
                    candidates.append(os.path.join(dirpath, filename))

    if not candidates:
        raise FileNotFoundError(
            "No CSV/XLS/XLSX file found. Add the Kaggle dataset to the notebook "
            "or set DATASET_PATH manually."
        )

    # First priority: files that actually contain the required raw dataset columns.
    valid_raw_files = [path for path in candidates if file_has_expected_columns(path)]
    if valid_raw_files:
        keywords = ["ev", "charging", "financial", "risk", "behavior"]
        preferred = [
            path for path in valid_raw_files
            if any(word in os.path.basename(path).lower() for word in keywords)
        ]
        return sorted(preferred or valid_raw_files)[0]

    # Fallback: use filename keywords if header check cannot be completed.
    keywords = ["ev", "charging", "financial", "risk", "behavior"]
    preferred = [
        path for path in candidates
        if any(word in os.path.basename(path).lower() for word in keywords)
    ]

    selected = sorted(preferred or candidates)[0]
    return selected

def load_raw_dataset():
    path = find_dataset_file()
    print(f"Loading dataset from: {path}")

    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        raise ValueError("Unsupported file type. Use CSV, XLSX, or XLS.")

    print(f"Raw dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df

def clean_dataset(df):
    """
    Basic cleaning for this EV charging dataset:
    - strips column names and string values
    - removes duplicate rows
    - converts known numeric columns
    - converts impossible negative values into missing values
    - standardizes binary columns where needed
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    df = df.drop_duplicates()

    # Strip text columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

    # Standardize binary columns if they came in as text before numeric conversion
    binary_maps = {
        "yes": 1, "y": 1, "true": 1, "high": 1, "risk": 1, "default": 1, "1": 1,
        "no": 0, "n": 0, "false": 0, "low": 0, "non-risk": 0, "non risk": 0, "0": 0
    }
    for col in ["Loan_Taken", "High_Default_Risk"]:
        if col in df.columns and df[col].dtype == "object":
            df[col] = df[col].astype(str).str.lower().map(binary_maps)

    known_numeric = [
        "User_ID", "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month", "Loan_Taken",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "High_Default_Risk", "Charging_Efficiency_Index"
    ]
    for col in known_numeric:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Treat impossible negative values as missing
    non_negative_cols = [
        "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "Charging_Efficiency_Index"
    ]
    for col in non_negative_cols:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    # Keep app score in a practical range if the column exists
    if "App_Usage_Score" in df.columns:
        df.loc[(df["App_Usage_Score"] < 0) | (df["App_Usage_Score"] > 10), "App_Usage_Score"] = np.nan

    return df

def make_output_dir(name):
    output_dir = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else ".", name)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output folder: {output_dir}")
    return output_dir

def split_features_target(df, target_col, drop_cols=None):
    if drop_cols is None:
        drop_cols = []
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available columns: {list(df.columns)}")

    data = df.dropna(subset=[target_col]).copy()

    # Force target to integer binary if classification
    if target_col == "High_Default_Risk":
        data = data[data[target_col].isin([0, 1])].copy()
        data[target_col] = data[target_col].astype(int)

    y = data[target_col]
    X = data.drop(columns=[target_col] + [c for c in drop_cols if c in data.columns])
    return X, y, data

def get_feature_types(X):
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    return numeric_cols, categorical_cols

def make_preprocessor(X, scale_numeric=True):
    numeric_cols, categorical_cols = get_feature_types(X)

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    try:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
    except TypeError:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ],
        remainder="drop"
    )
    return preprocessor, numeric_cols, categorical_cols

def get_transformed_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = list(numeric_cols)
    if categorical_cols:
        try:
            ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
            cat_names = list(ohe.get_feature_names_out(categorical_cols))
        except Exception:
            cat_names = []
            for col in categorical_cols:
                cat_names.append(col)
        feature_names.extend(cat_names)
    return feature_names

def save_table(df, output_dir, filename):
    path = os.path.join(output_dir, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

def save_figure(output_dir, filename):
    path = os.path.join(output_dir, filename)
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def style_plot(title, subtitle=None, xlabel=None, ylabel=None):
    plt.title(title, fontsize=15, fontweight="bold", pad=18)
    if subtitle:
        plt.suptitle(subtitle, fontsize=10, y=0.94, color="#3b4f6b")
    if xlabel:
        plt.xlabel(xlabel, fontsize=11)
    if ylabel:
        plt.ylabel(ylabel, fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.gca().set_axisbelow(True)

def print_dataset_summary(df):
    print("Cleaned dataset shape:", df.shape)
    print("\nColumns:")
    print(list(df.columns))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(20))


# ============================================================
# RQ4: Feature importance for high default-risk prediction
# ============================================================

from sklearn.ensemble import RandomForestClassifier

output_dir = make_output_dir("rq4_feature_importance")

df_raw = load_raw_dataset()
df = clean_dataset(df_raw)
print_dataset_summary(df)

target_col = "High_Default_Risk"
X, y, data = split_features_target(df, target_col, drop_cols=["User_ID"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

preprocessor, numeric_cols, categorical_cols = make_preprocessor(X_train, scale_numeric=False)
model = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_score = pipe.predict_proba(X_test)[:, 1]
print("Model check:")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_score), 4))

feature_names = get_transformed_feature_names(
    pipe.named_steps["preprocessor"],
    numeric_cols,
    categorical_cols
)

importances = pipe.named_steps["model"].feature_importances_

# Direction based on correlation between transformed feature values and target
X_train_transformed = pipe.named_steps["preprocessor"].transform(X_train)
directions = []

for i in range(len(feature_names)):
    values = X_train_transformed[:, i]
    if np.std(values) == 0:
        corr = 0
    else:
        corr = np.corrcoef(values, y_train)[0, 1]
        if np.isnan(corr):
            corr = 0
    if corr > 0:
        directions.append("Higher/present → higher risk")
    elif corr < 0:
        directions.append("Higher/present → lower risk")
    else:
        directions.append("No clear direction")

results = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances,
    "Direction_of_Association": directions
})

results = results.sort_values("Importance", ascending=False).reset_index(drop=True)
results.insert(0, "Rank", np.arange(1, len(results) + 1))

top_results = results.head(15).copy()
top_results["Importance"] = top_results["Importance"].round(4)

display(top_results)
save_table(top_results, output_dir, "RQ4_feature_importance_table.csv")

top = top_results.iloc[0]
print(
    f"Actual answer for RQ4: The most important predictor is {top['Feature']} "
    f"with feature importance = {top['Importance']}."
)

# Figure 4: Horizontal feature importance plot
plot_df = top_results.sort_values("Importance", ascending=True)

plt.figure(figsize=(11, 8))
bars = plt.barh(plot_df["Feature"], plot_df["Importance"], color="#2f72c4")

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.003, bar.get_y() + bar.get_height()/2, f"{width:.3f}", va="center", fontsize=9)

plt.xlabel("Feature importance")
plt.title("Figure 4. Most important predictors of high default risk", fontsize=15, fontweight="bold", pad=18)
plt.grid(axis="x", linestyle="--", alpha=0.35)
plt.figtext(
    0.5, -0.02,
    "Note: Larger values indicate stronger contribution to the prediction in the Random Forest model.",
    ha="center", fontsize=9
)
plt.tight_layout()

save_figure(output_dir, "RQ4_feature_importance_figure.pdf")
plt.show()
